In [1]:
from march.module import DMC
from march.ops.grid import create_voxel_grid

import torch
import open3d as o3d
import open3d.core as o3c
import numpy as np
import plotly.graph_objects as go

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Object

In [2]:
sphere_mesh = o3d.t.geometry.TriangleMesh.create_sphere(radius=1.0, resolution=20)

print("# vertices:", len(sphere_mesh.vertex.positions))
print("# triangles:", len(sphere_mesh.triangle.indices))

# vertices: 762
# triangles: 1520


# Voxel Grid

In [3]:
resolution = 16
bounds = [[-1.5, 1.5], [-1.5, 1.5], [-1.5, 1.5]]
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
grid_vertices, cubes = create_voxel_grid(
    res_x=resolution,
    res_y=resolution,
    res_z=resolution,
    bounds=bounds,
)

print("grid vertices:", grid_vertices.shape)
print("cubes:", cubes.shape)

grid vertices: (4913, 3)
cubes: (4096, 8)


# Sign Distance

In [5]:
scene = o3d.t.geometry.RaycastingScene()
scene.add_triangles(sphere_mesh)

0

In [6]:
queries = o3c.Tensor(grid_vertices, dtype=o3c.float32)
print("queries:", queries.shape)

queries: SizeVector[4913, 3]


In [7]:
distances = scene.compute_signed_distance(queries)

print("distances:", distances.shape)
print("distances (min, max):", distances.min().item(), distances.max().item())

distances: SizeVector[4913]
distances (min, max): -0.9938629269599915 1.5982102155685425


In [8]:
print(type(distances))

<class 'open3d.cuda.pybind.core.Tensor'>


# Reconstruction

In [9]:
grid_vertices = torch.from_numpy(grid_vertices).to(device)
cubes = torch.from_numpy(cubes).to(device)
distances = torch.from_numpy(distances.cpu().numpy()).to(device)

In [10]:
print(f"grid_vertices dtype: {grid_vertices.dtype}, device: {grid_vertices.device}")
print(f"cubes dtype: {cubes.dtype}, device: {cubes.device}")
print(f"distances dtype: {distances.dtype}, device: {distances.device}")

grid_vertices dtype: torch.float32, device: cuda:0
cubes dtype: torch.int32, device: cuda:0
distances dtype: torch.float32, device: cuda:0


In [11]:
mc = DMC()
iso = 0.0

vertices, triangles = mc(
    grid_vertices=grid_vertices, 
    cubes=cubes, 
    values=distances, 
    iso=iso
)

In [12]:
print(f"# vertices: {vertices.shape[0]}, # triangles: {triangles.shape[0]}")
print(f"vertices dtype: {vertices.dtype}, device: {vertices.device}")
print(f"triangles dtype: {triangles.dtype}, device: {triangles.device}")

# vertices: 534, # triangles: 1064
vertices dtype: torch.float32, device: cuda:0
triangles dtype: torch.int64, device: cuda:0


In [14]:
import plotly.graph_objects as go

# Create figure
fig = go.Figure()

# Create color array: red if value > iso, blue otherwise
colors = ['red' if v > iso else 'blue' for v in distances.cpu().numpy()]

grids_numpy = grid_vertices.detach().cpu().numpy()
verts_numpy = vertices.detach().cpu().numpy()
faces_numpy = triangles.detach().cpu().numpy()

# Add grid points as scatter plot
# fig.add_trace(go.Scatter3d(
#     x=grids_numpy[:, 0],
#     y=grids_numpy[:, 1],
#     z=grids_numpy[:, 2],
#     mode='markers',
#     marker=dict(size=5, color=colors, opacity=0.8),
#     name='Grid Points'
# ))

# Add mesh as wireframe
# First, create the mesh surface with low opacity
fig.add_trace(go.Mesh3d(
    x=verts_numpy[:, 0],
    y=verts_numpy[:, 1],
    z=verts_numpy[:, 2],
    i=faces_numpy[:, 0],
    j=faces_numpy[:, 1],
    k=faces_numpy[:, 2],
    opacity=0.1,
    color='lightblue',
    showlegend=False
))

# Add wireframe edges
edges_x = []
edges_y = []
edges_z = []
for i, j, k in faces_numpy:
    # Edge 1: vertex i to j
    edges_x.extend([verts_numpy[i, 0], verts_numpy[j, 0], None])
    edges_y.extend([verts_numpy[i, 1], verts_numpy[j, 1], None])
    edges_z.extend([verts_numpy[i, 2], verts_numpy[j, 2], None])
    # Edge 2: vertex j to k
    edges_x.extend([verts_numpy[j, 0], verts_numpy[k, 0], None])
    edges_y.extend([verts_numpy[j, 1], verts_numpy[k, 1], None])
    edges_z.extend([verts_numpy[j, 2], verts_numpy[k, 2], None])
    # Edge 3: vertex k to i
    edges_x.extend([verts_numpy[k, 0], verts_numpy[i, 0], None])
    edges_y.extend([verts_numpy[k, 1], verts_numpy[i, 1], None])
    edges_z.extend([verts_numpy[k, 2], verts_numpy[i, 2], None])

fig.add_trace(go.Scatter3d(
    x=edges_x,
    y=edges_y,
    z=edges_z,
    mode='lines',
    line=dict(color='darkblue', width=2),
    name='Mesh Edges',
    showlegend=True
))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data'
    ),
    title='Grid Points and Marching Cubes Mesh (Wireframe)',
    width=800,
    height=800
)

fig.show()


In [15]:
# Export to OBJ

o3d_mesh = o3d.geometry.TriangleMesh()
o3d_mesh.vertices = o3d.utility.Vector3dVector(verts_numpy)
o3d_mesh.triangles = o3d.utility.Vector3iVector(faces_numpy)
o3d.io.write_triangle_mesh("marching_cubes_sphere.obj", o3d_mesh)

True